# HI 5304 — APIs & JSON in Python (Jupyter Notebook)

*Created: 2026-01-25*

This notebook teaches the essentials of working with **APIs** and **JSON** in Python using **Jupyter** (ideal for GitHub Codespaces). It is **read-only** and uses public endpoints.


## Learning goals
- Make a GET request to a public API
- Understand JSON → Python dictionaries/lists
- Handle parameters, headers, and common errors
- Normalize JSON into a `pandas` DataFrame
- Practice with short exercises


## 0. Setup
Most Codespaces images already include `requests`. If you need it, run the install cell once.


In [ ]:
# Optional: run once if needed
# %pip install requests pandas

import requests
import json
import pandas as pd


## 1. What is an API?
**API** (Application Programming Interface) endpoints let software exchange data.

Most modern APIs return data as **JSON** (JavaScript Object Notation), which maps cleanly to Python:
- JSON object → Python `dict`
- JSON array → Python `list`


## 2. Your first API call (GitHub)
We'll call GitHub’s public API to fetch metadata about the class repo.


In [4]:
import requests
import json
import pandas as pd


In [5]:
url = "https://api.github.com/repos/Health-informatics-Data-Analytics/hi5304-notebooks"

r = requests.get(url, headers={"User-Agent": "hi5304"})
r.raise_for_status()  # raises an error for 4xx/5xx responses

data = r.json()  # JSON -> Python dict
type(data), list(data.keys())[:10]


(dict,
 ['id',
  'node_id',
  'name',
  'full_name',
  'private',
  'owner',
  'html_url',
  'description',
  'fork',
  'url'])

### Inspect a few fields


In [6]:
data["full_name"], data["open_issues_count"], data["stargazers_count"], data["updated_at"]


('Health-informatics-Data-Analytics/hi5304-notebooks',
 0,
 1,
 '2026-01-25T16:02:35Z')

### Pretty-print JSON (first 1200 characters)
This helps you *see* nesting and field names.


In [7]:
print(json.dumps(data, indent=2)[:1200])


{
  "id": 1129982692,
  "node_id": "R_kgDOQ1oq5A",
  "name": "hi5304-notebooks",
  "full_name": "Health-informatics-Data-Analytics/hi5304-notebooks",
  "private": false,
  "owner": {
    "login": "Health-informatics-Data-Analytics",
    "id": 253527654,
    "node_id": "O_kgDODxyGZg",
    "avatar_url": "https://avatars.githubusercontent.com/u/253527654?v=4",
    "gravatar_id": "",
    "url": "https://api.github.com/users/Health-informatics-Data-Analytics",
    "html_url": "https://github.com/Health-informatics-Data-Analytics",
    "followers_url": "https://api.github.com/users/Health-informatics-Data-Analytics/followers",
    "following_url": "https://api.github.com/users/Health-informatics-Data-Analytics/following{/other_user}",
    "gists_url": "https://api.github.com/users/Health-informatics-Data-Analytics/gists{/gist_id}",
    "starred_url": "https://api.github.com/users/Health-informatics-Data-Analytics/starred{/owner}{/repo}",
    "subscriptions_url": "https://api.github.com/users

## 3. Nested JSON
Many APIs (including FHIR) use nested objects.
Here, `owner` is a nested dictionary.


In [8]:
type(data["owner"]), data["owner"]["login"], data["owner"]["html_url"]


(dict,
 'Health-informatics-Data-Analytics',
 'https://github.com/Health-informatics-Data-Analytics')

### Safer access with `.get()`
APIs can change. `.get()` avoids `KeyError` if a field is missing.


In [9]:
license_info = data.get("license")  # could be None
language = data.get("language", "Not specified")

license_info, language


(None, 'Jupyter Notebook')

## 4. Common request components
- **URL**: the endpoint
- **Query parameters**: e.g., `?per_page=10`
- **Headers**: metadata (often auth + content type)
- **Timeouts**: prevent hanging forever


### Example: parameters (`params=`)
Fetch the 10 most recent issues (if any).


In [10]:
issues_url = "https://api.github.com/repos/Health-informatics-Data-Analytics/hi5304-notebooks/issues"

r = requests.get(
    issues_url,
    params={"per_page": 10, "state": "open"},
    headers={"User-Agent": "hi5304"},
    timeout=10
)
r.raise_for_status()
issues = r.json()

type(issues), len(issues)


(list, 0)

### Normalize a list of JSON objects into a DataFrame
If there are no open issues, the DataFrame will just be empty.


In [11]:
df_issues = pd.json_normalize(issues)
df_issues.head()


""


## 5. Error handling patterns (teachable + practical)
Students should learn how to interpret errors and respond safely.


In [12]:
def safe_get(url: str, *, params=None, headers=None, timeout=10):
    try:
        resp = requests.get(url, params=params, headers=headers, timeout=timeout)
        resp.raise_for_status()
        return resp
    except requests.exceptions.Timeout:
        print("⏱️ Timeout: the server took too long to respond.")
    except requests.exceptions.HTTPError as e:
        print(f"🚫 HTTP error: {e} | status={getattr(resp, 'status_code', None)}")
        try:
            print("Response body (first 300 chars):", resp.text[:300])
        except Exception:
            pass
    except requests.exceptions.RequestException as e:
        print("❗ Network/request error:", e)

resp = safe_get("https://api.github.com/rate_limit", headers={"User-Agent":"hi5304"})
rate_data = resp.json() if resp else None
rate_data and rate_data["rate"]


{'limit': 60,
 'remaining': 58,
 'reset': 1769367726,
 'used': 2,
 'resource': 'core'}

## 6. Rate limits (important for real-world APIs)
Many APIs limit requests per hour/day. GitHub exposes rate limit information.


In [13]:
if rate_data:
    core = rate_data["resources"]["core"]
    print("Limit:", core["limit"])
    print("Remaining:", core["remaining"])
    print("Resets (unix time):", core["reset"])


Limit: 60
Remaining: 58
Resets (unix time): 1769367726


## 7. API keys & secrets (DO NOT commit keys)
For course work, the safest pattern is:
- Store keys in **Codespaces Secrets** or an untracked `.env`
- Read them via environment variables

Example (won't print the full key):


In [14]:
import os

# Example only: set a Codespaces secret like OPENWEATHER_API_KEY (do not hardcode keys)
api_key = os.getenv("OPENWEATHER_API_KEY")

if api_key:
    print("✅ API key loaded (first 4 chars):", api_key[:4] + "...")
else:
    print("ℹ️ No API key found. That's okay for this notebook's public endpoints.")


ℹ️ No API key found. That's okay for this notebook's public endpoints.


## 8. Mini-project: pull commit activity and summarize
We'll fetch recent commits and build a tidy table.


In [15]:
commits_url = "https://api.github.com/repos/Health-informatics-Data-Analytics/hi5304-notebooks/commits"

resp = safe_get(commits_url, params={"per_page": 15}, headers={"User-Agent":"hi5304"})
commits = resp.json() if resp else []

rows = []
for c in commits:
    rows.append({
        "sha": c.get("sha", "")[:10],
        "author": (c.get("commit", {}).get("author", {}) or {}).get("name"),
        "date": (c.get("commit", {}).get("author", {}) or {}).get("date"),
        "message": (c.get("commit", {}) or {}).get("message", "").splitlines()[0][:80],
    })

df_commits = pd.DataFrame(rows)
df_commits.head(10)


,sha,author,date,message
0,5f0ffc3068,Patrick Dunn,2026-01-25T16:02:26Z,learning r datasets
1,b8635ca6ef,Patrick Dunn,2026-01-25T14:42:05Z,R examples
2,774df7182b,Patrick Dunn,2026-01-25T00:40:40Z,libraries
3,5dfa460960,Patrick Dunn,2026-01-25T00:35:10Z,updated
4,d64f5420c4,Patrick Dunn,2026-01-24T23:52:52Z,rebuild
5,59bd04528f,Patrick Dunn,2026-01-24T23:26:17Z,updating dockerfile
6,38990ebb11,Patrick Dunn,2026-01-24T22:58:11Z,requirements
7,64d16db4c9,Patrick Dunn,2026-01-24T22:56:19Z,learning
8,ad48fd8499,Patrick Dunn,2026-01-24T21:01:40Z,Updated dockerfile
9,40a6165898,Patrick Dunn,2026-01-24T18:47:53Z,practice


### Quick summary


In [16]:
df_commits["author"].value_counts(dropna=False).head(10)


author
Patrick Dunn    15
Name: count, dtype: int64

## 9. Exercises (for students)
1) Change `per_page` from 15 to 5 in the commits request. What changes?
2) Add a column called `message_len` with the length of each commit message.
3) Using the `issues` endpoint, request `state=all` and find how many issues are returned.
4) (Challenge) Create a function that takes an owner + repo name and returns a DataFrame summary.


In [ ]:
# Exercise 2 starter: add message_len
df_commits["message_len"] = df_commits["message"].fillna("").str.len()
df_commits[["sha","author","date","message_len","message"]].head(10)


## 10. Extension: swap in a health-focused API
Once this pattern is comfortable, we can repeat the same workflow with public health APIs like:
- CDC public datasets (Socrata)
- OpenFDA
- U.S. Census

Tell me which one you want, and I’ll provide a drop-in section.


Example: OpenFDA Drug Adverse Events API
What this API represents (context for students)

OpenFDA provides access to FDA Adverse Event Reporting System (FAERS) data:

Reports of side effects, medication errors, and adverse reactions

Used in pharmacovigilance and post-market surveillance

Real-world, messy, high-volume public health data

🔗 Conceptually similar to:

Disease surveillance feeds

Safety signal monitoring systems

Real-world evidence pipelines

Step 1: Define the endpoint

This query asks:

“Give me the 10 most recent adverse event reports involving metformin.”

In [17]:
import requests
import json
import pandas as pd

url = "https://api.fda.gov/drug/event.json"

params = {
    "search": "patient.drug.medicinalproduct:metformin",
    "limit": 10
}

r = requests.get(url, params=params, timeout=10)
r.raise_for_status()

data = r.json()
type(data), data.keys()


(dict, dict_keys(['meta', 'results']))

Step 2: Inspect the structure

In [18]:
print(json.dumps(data, indent=2)[:1500])


{
  "meta": {
    "disclaimer": "Do not rely on openFDA to make decisions regarding medical care. While we make every effort to ensure that data is accurate, you should assume all results are unvalidated. We may limit or otherwise restrict your access to the API in line with our Terms of Service.",
    "terms": "https://open.fda.gov/terms/",
    "license": "https://open.fda.gov/license/",
    "last_updated": "2025-10-30",
    "results": {
      "skip": 0,
      "limit": 10,
      "total": 413238
    }
  },
  "results": [
    {
      "safetyreportversion": "2",
      "safetyreportid": "10003641",
      "primarysourcecountry": "US",
      "occurcountry": "US",
      "transmissiondateformat": "102",
      "transmissiondate": "20150720",
      "reporttype": "1",
      "serious": "2",
      "receivedateformat": "102",
      "receivedate": "20140312",
      "receiptdateformat": "102",
      "receiptdate": "20150312",
      "fulfillexpeditecriteria": "2",
      "companynumb": "US-ABBVIE-13P-1

Step 3: Extract the results

In [19]:
events = data["results"]
len(events), type(events)


(10, list)

Step 4: Normalize JSON into a DataFrame

In [20]:
rows = []

for e in events:
    patient = e.get("patient", {})
    drugs = patient.get("drug", [])
    reactions = patient.get("reaction", [])

    rows.append({
        "safety_report_id": e.get("safetyreportid"),
        "received_date": e.get("receivedate"),
        "serious": e.get("serious"),
        "drug_count": len(drugs),
        "reaction_count": len(reactions),
        "reactions": "; ".join(
            [r.get("reactionmeddrapt", "") for r in reactions[:3]]
        )
    })

df = pd.DataFrame(rows)
df


,safety_report_id,received_date,serious,drug_count,reaction_count,reactions
0,10003641,20140312,2,6,1,Medication residue present
1,10003642,20140312,1,11,1,Osteomyelitis
2,10003647,20140312,1,6,2,Erysipelas; Sepsis
3,10003803,20140312,2,4,3,Balance disorder; Gait disturbance; Gait distu...
4,10003915,20140312,2,11,1,Weight increased
5,10003959,20140312,2,11,3,Splenomegaly; Dyspnoea; Insomnia
6,10003993,20140312,1,9,2,Metastases to bone; Osteonecrosis of jaw
7,10003998,20140312,2,15,11,Feeling abnormal; Pneumonia; Suicidal ideation
8,10004006,20140312,2,10,5,Nausea; Vomiting; Abdominal pain upper
9,10004017,20140312,2,6,1,Epistaxis


CDC (Socrata): Population health / surveillance-style indicators

Example dataset: CDC U.S. Chronic Disease Indicators (Socrata dataset ID: hksd-2xuw).

In [21]:
import requests
import pandas as pd

# Socrata SODA endpoint pattern:
# https://data.cdc.gov/resource/<dataset_id>.json
url = "https://data.cdc.gov/resource/hksd-2xuw.json"

params = {
    "$select": "yearstart,locationdesc,topic,question,datavalueunit,datavalue",
    "$where": "yearstart >= 2020 AND locationdesc = 'Texas'",
    "$limit": 25
}

r = requests.get(url, params=params, timeout=20)
r.raise_for_status()
rows = r.json()

df = pd.DataFrame(rows)
df.head(10)


,yearstart,locationdesc,topic,question,datavalueunit,datavalue
0,2020,Texas,Asthma,"Asthma mortality among all people, underlying ...",Number,136
1,2020,Texas,Oral Health,Visited dentist or dental clinic in the past y...,%,54.3
2,2020,Texas,Tobacco,Current cigarette smoking among adults,%,9.3
3,2020,Texas,Cardiovascular Disease,Cerebrovascular disease (stroke) mortality amo...,"cases per 100,000",NaN
4,2020,Texas,Health Status,Average recent physically unhealthy days among...,Number,2.9
5,2021,Texas,Asthma,Current asthma among adults,%,9
6,2021,Texas,Cardiovascular Disease,Coronary heart disease mortality among all peo...,Number,16921
7,2021,Texas,Diabetes,"Diabetes mortality among all people, underlyin...",Number,9567
8,2021,Texas,Disability,Adults with any disability,%,29.6
9,2021,Texas,Immunization,Influenza vaccination among adults,%,29.1


U.S. Census API: SDOH / demographics (ACS)

Example: ACS 5-year (total population variable B01001_001E) using the Census “examples” endpoint for 2022.

In [22]:
import requests
import pandas as pd
import os

# Census API base (ACS 5-year, 2022 example)
base = "https://api.census.gov/data/2022/acs/acs5"

# Optional: Census API key (recommended). If you don't have one, this may still work but can be rate-limited.
key = os.getenv("CENSUS_API_KEY")

params = {
    "get": "NAME,B01001_001E",   # NAME + total population estimate
    "for": "state:*"            # all states
}
if key:
    params["key"] = key

r = requests.get(base, params=params, timeout=20)
r.raise_for_status()
data = r.json()

# First row is headers
cols = data[0]
rows = data[1:]

df = pd.DataFrame(rows, columns=cols)
df["B01001_001E"] = pd.to_numeric(df["B01001_001E"], errors="coerce")
df.sort_values("B01001_001E", ascending=False).head(10)


,NAME,B01001_001E,state
4,California,39356104,06
43,Texas,29243342,48
9,Florida,21634529,12
32,New York,19994379,36
38,Pennsylvania,12989208,42
13,Illinois,12757634,17
35,Ohio,11774683,39
10,Georgia,10722325,13
33,North Carolina,10470214,37
22,Michigan,10057921,26


CMS Data API: Utilization / cost / quality measures (data.cms.gov)

CMS provides dataset-specific API docs pages that include the dataset UUID(s) you query via /dataset/{uuid}/data.

Example dataset: Medicare Inpatient Hospitals – by Provider
Use the 2023 version UUID shown on the CMS API docs page: ad17abc4-0e93-4828-bba5-e8f39ddafbdb.

In [23]:
import requests
import pandas as pd

uuid_2023 = "ad17abc4-0e93-4828-bba5-e8f39ddafbdb"
url = f"https://data.cms.gov/data-api/v1/dataset/{uuid_2023}/data"

params = {
    "limit": 25,
    # You can filter with "query" for simple text matching.
    # For structured filtering/column selection, check the dataset's api-docs page.
    "query": "Texas"
}

r = requests.get(url, params=params, timeout=30)
r.raise_for_status()
payload = r.json()

# CMS returns objects; often the records are in payload (list) or payload['data'] depending on endpoint/version
rows = payload if isinstance(payload, list) else payload.get("data", payload)

df = pd.DataFrame(rows)
df.head(10)


,Rndrng_Prvdr_CCN,Rndrng_Prvdr_Org_Name,Rndrng_Prvdr_St,Rndrng_Prvdr_City,Rndrng_Prvdr_Zip5,Rndrng_Prvdr_State_Abrvtn,Rndrng_Prvdr_State_FIPS,Rndrng_Prvdr_RUCA,Rndrng_Prvdr_RUCA_Desc,Tot_Benes,...,Bene_CC_PH_Diabetes_V2_Pct,Bene_CC_PH_HF_NonIHD_V2_Pct,Bene_CC_PH_Hyperlipidemia_V2_Pct,Bene_CC_PH_Hypertension_V2_Pct,Bene_CC_PH_IschemicHeart_V2_Pct,Bene_CC_PH_Osteoporosis_V2_Pct,Bene_CC_PH_Parkinson_V2_Pct,Bene_CC_PH_Arthritis_V2_Pct,Bene_CC_PH_Stroke_TIA_V2_Pct,Bene_Avg_Risk_Scre
0,010001,Southeast Health Medical Center,1108 Ross Clark Circle,Dothan,36301,AL,01,2,Metropolitan area high commuting: primary flow...,3088,...,0.4957901554,0.4381476684,0.75,0.75,0.5,0.1444300518,0.0272020725,0.6091321244,0.2678108808,2.0048648
1,010005,Marshall Medical Centers South Campus,2505 U S Highway 431 North,Boaz,35957,AL,01,4,Micropolitan area core: primary flow within an...,1123,...,0.4336598397,0.482635797,0.75,0.75,0.4737310775,0.1665182547,0.0471950134,0.606411398,0.2644701692,1.723296633
2,010006,North Alabama Medical Center,1701 Veterans Drive,Florence,35630,AL,01,1,Metropolitan area core: primary flow within an...,2634,...,0.4753227031,0.4202733485,0.75,0.75,0.5170842825,0.1924829157,0.0584662111,0.6511009871,0.2141230068,1.910259056
3,010007,Mizell Memorial Hospital,702 N Main St,Opp,36467,AL,01,7,Small town core: primary flow within an urban ...,252,...,0.5079365079,0.5238095238,0.75,0.75,0.4404761905,0.1111111111,0.0753968254,0.6031746032,0.1944444444,1.9350848218
4,010008,Crenshaw Community Hospital,101 Hospital Circle,Luverne,36049,AL,01,3,Metropolitan area low commuting: primary flow ...,89,...,0.4494382022,0.3033707865,0.75,0.75,0.4269662921,0.202247191,0.0337078652,0.5505617978,0.1797752809,1.6179516343
5,010011,St Vincent's East,50 Medical Park East Drive,Birmingham,35235,AL,01,1,Metropolitan area core: primary flow within an...,1943,...,0.4621718991,0.4307771487,0.75,0.75,0.5285640762,0.117858981,0.0349974267,0.5054040144,0.2120432321,2.098970215
6,010012,Dekalb Regional Medical Center,200 Med Center Drive,Fort Payne,35968,AL,01,10,Rural areas: primary flow to a tract outside a...,454,...,0.4713656388,0.4427312775,0.75,0.75,0.5,0.0991189427,0.0440528634,0.5969162996,0.1233480176,1.637160207
7,010016,Shelby Baptist Medical Center,1000 First Street North,Alabaster,35007,AL,01,1,Metropolitan area core: primary flow within an...,1143,...,0.4759405074,0.5704286964,0.75,0.75,0.6351706037,0.1609798775,0.0472440945,0.5433070866,0.2257217848,2.0214035432
8,010018,Uab Callahan Eye Hospital Authority,1720 University Blvd Ste 305,Birmingham,35233,AL,01,1,Metropolitan area core: primary flow within an...,11,...,0.1818181818,0.0909090909,0.6363636364,0.75,0.2727272727,0.0909090909,0,0.6363636364,0.0909090909,0.7435454545
9,010019,Helen Keller Memorial Hospital,1300 South Montgomery Avenue,Sheffield,35660,AL,01,1,Metropolitan area core: primary flow within an...,1152,...,0.4670138889,0.4175347222,0.75,0.75,0.4210069444,0.1701388889,0.0546875,0.6545138889,0.1710069444,1.846072343
